# 03 - Calibration: the YAML declares the parameters, IPOPT finds them

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andoludo/mpctrano/blob/main/tutorials/03_calibration_from_yaml.ipynb)

**Learning objectives**

1. Declare the parameters to calibrate in the Trano YAML itself (`{value, parameter: true, min, max}`), keep it
   the single source of truth, and see which parameters of the CasADi model each field moves.
2. Calibrate the **closed-loop** model (the controllers of the YAML inside, no control input) on monitoring data
   a real building would give: a noisy air temperature and an electricity meter, no valve positions.
3. Read how IPOPT solves it: the three fields as decision variables, the closed-loop model simulated with CVODES
   inside a weighted least squares with priors, the gradient from the adjoint sensitivities of the integrator.
4. Compare the model before and after: the temperatures and the energy on the training days and on two
   validation days, and the recovered values against the true ones.

The true building, its data and the calibration come from `tutorials/cache/calibration/` (`make cache`); the
calibration is solved live when the notebook does not run in quick mode (a few minutes).

**Runtimes** (this container): one regeneration of the model by Trano and rumoca about 15 s (one per declared
parameter, for the sensitivities); the calibration of three days of hourly data, 38 states, two to five minutes.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: fetch the tutorials, install mpctrano and rumoca (a no-op elsewhere)
    import subprocess

    subprocess.run(  # a fixed command; the clone is skipped when it exists
        ["git", "clone", "-q", "--depth", "1", "https://github.com/andoludo/mpctrano", "/content/mpctrano"], check=False
    )
    sys.path.insert(0, "/content/mpctrano")
    from tutorials import _colab

    _colab.setup()

In [ ]:
import pandas as pd
from IPython.display import display

from mpctrano import CalibrationSettings, Calibrator
from mpctrano.calibration import evaluate, simulate_fit
from mpctrano.notebook import quick_mode, timed
from mpctrano.plot import plot_fit
from tutorials import _calibration as study

pd.set_option("display.width", 200)
print(f"quick mode: {quick_mode()}")

## 1. The parameters, declared in the YAML

`tests/models/single_zone_calibration.yaml` is the heat pump house of tutorial 01 with its cavity wall spelled out
and three fields annotated: the conductivity of the insulation, its thickness and the infiltration rate.
`CalibrationSpec` reads them by path (the convention of `trano.runtime`) and writes the flat copy Trano reads.

In [ ]:
spec = study.spec()
text = study.YAML.read_text()
print(text[: text.index("    id: SPACE:001")])
display(pd.DataFrame([parameter.model_dump(by_alias=True) for parameter in spec.parameters]).set_index("path"))

Which parameters of the CasADi model a field moves is **measured**: the model is regenerated with the field raised
by 20 % and the exponent of every parameter that moved is read. The conductivity moves the conductance of the
insulation layer of the three walls (exponent 1), the thickness moves it with -1 and the capacities of the
neighbouring nodes a little, the infiltration rate is the `ACH` of the zone.

The table also says what *not* to calibrate together: the conductivity and the thickness enter the model through
the same conductance (`k / x`, exponents +1 and -1), the capacities they move a little apart; no data on the air
temperature tells them apart. The study keeps the thickness as drawn (a drawing is a measurement) and
calibrates the conductivity and the infiltration rate.

In [ ]:
for group in study.declared_groups():
    moved = {
        name: round(exponent, 3)
        for name, exponent in group.exponents.items()
        if not name.startswith(("externalwall_1", "externalwall_2"))
    }
    print(f"{group.name}: value {group.value:g} in [{group.lower:g}, {group.upper:g}] -> {moved} (and the other walls)")
groups = study.groups()
print("calibrated:", [group.name for group in groups])

## 2. The true building and its monitoring data

The *true* building is the same YAML with other values of the three parameters: a worse insulation, thinner than
drawn, and a leakier house. It is run by its own controllers (the closed-loop `FastModel`: thermostat, heating
curve, buffer control) over five cold days of January, after two days of spin-up; the monitoring system records
the weather, the air temperature with 0.1 K of noise and the electricity meter with 20 Wh of noise. It does
**not** record the valves or the compressor: the model being calibrated has its controllers inside.

In [ ]:
model = study.model()
print(
    "states",
    model.n_states,
    "| controls",
    model.control_names,
    "| measured",
    model.comfort_states,
    "| meters",
    list(model.meters),
)
truth = pd.Series(spec.nominal, name="assumed (YAML)").to_frame()
truth["true value"] = pd.Series(study.TRUTH).reindex(truth.index).fillna(truth["assumed (YAML)"])
display(truth)
data, true_states = study.recorded()
training, validation = study.split(data)
print(f"{len(training) - 1} training hours from {training.index[0]}, {len(validation) - 1} validation hours")
display(data[[*model.comfort_states, *model.meters]].describe().round(2).T)

## 3. Before: the model with the values of the YAML

An open-loop simulation of the model from the first measurement, with the assumed parameters: the house loses
less heat than the true one, so it stays warmer and draws less electricity.

In [ ]:
before = simulate_fit(model, training)
metrics_before, _ = evaluate(model, training)
display(plot_fit(before, "before calibration: training days (dotted: measured)"))
display(metrics_before.frame().round(3))

## 4. How IPOPT calibrates it

The calibration is one nonlinear programme over the whole training period, solved by **single shooting**
(`CalibrationSettings.method`, chosen automatically for a closed-loop model):

* **decision variables**: one per declared parameter, `log(field / value)`, bounded by `log(min / value)` and
  `log(max / value)`; nothing else. The parameters of the model are expressions of the fields
  (`p = p0 * (field / value) ** e`, the exponents measured above; a conductivity and a thickness both move the
  conductance of the insulation layer, their factors compound).
* **the simulation inside the objective**: CVODES integrates the closed-loop model hour by hour, with the
  parameters of the candidate, from the start values of the Modelica model through two days of spin-up before
  the data (so that the walls, the water and the controller states at the start are those of the candidate, not
  of the nominal model) and then over the training days with their weather.
* **the gradient**: central differences of the objective with a step of 5 % of a field, 2n + 1 simulations of a
  second each; IPOPT's Hessian is a quasi-Newton (L-BFGS) one and it takes its full steps. A thermostat that
  flips one integration step earlier makes the objective jumpy at the scale of a tenth of a percent of a field:
  the adjoint sensitivities of the integrator and small finite differences read that noise and stall IPOPT
  short of the optimum, a wide step reads the slope of the valley through it and a sufficient-decrease test
  would refuse steps for the same jumps.
* **objective**: the squared errors of the simulated air temperature against the measurements (weighted by the
  sensor noise), of the energy metered over each hour against the meter (weighted by the meter noise), and the
  squared deviations of the fields from their prior (the YAML).

The models with control inputs (the MPC models of the other tutorials) are calibrated on their Radau
collocation instead, the states as decision variables and the dynamics as constraints: well conditioned over
long periods and poor initial parameters. Not here: the thermostat, the hysteresis of the heat pump controller
and the valve limits make the collocated dynamics of a closed-loop model non-smooth, and IPOPT crawls for
hundreds of iterations without converging; a simulation follows these switches in under a second, and three
parameters need a handful of iterations. The log below is IPOPT's own (one line per iteration: the objective,
the dual infeasibility, the step).

In [ ]:
if quick_mode():
    calibrated, result = study.calibrate()
    print("cached result:", result.status, "in", result.iterations, "iterations,", f"{result.solve_time:.0f} s")
else:
    with timed("calibration"):
        settings = CalibrationSettings(groups=groups, verbose=True, max_iterations=60)
        calibrated, result = Calibrator(model, settings).fit(training, validation)
print(
    f"{result.method}: {result.variables} decision variables, {result.iterations} iterations, "
    f"{result.solve_time:.0f} s, {result.status}"
)

## 5. After: the fit on the training and validation days, and what the two fields became

In [ ]:
recovered = pd.DataFrame({"assumed (YAML)": spec.nominal, "calibrated": result.factors, "true": truth["true value"]})
recovered["error %"] = 100 * (recovered["calibrated"] - recovered["true"]) / recovered["true"]
display(recovered.round(3))
display(result.metrics().round(3))

In [ ]:
x0 = study.initial_state(result)
after = simulate_fit(calibrated, training, x0)
display(plot_fit(after, "after calibration: training days (dotted: measured)"))
after_validation = simulate_fit(calibrated, validation)
before_validation = simulate_fit(model, validation)
display(plot_fit(after_validation, "after calibration: validation days (dotted: measured)"))

## 6. Identifiability: what the data can and cannot tell apart

The calibrated model reproduces the air temperature and the electricity of the true building on days it has not
seen, but the two fields did not land on their true values: a leakier house and a worse insulation are both a
loss proportional to the indoor-outdoor temperature difference. The data of a thermostat-controlled house fix the
**total** loss; how it splits between the walls and the infiltration is a flat valley of the objective, and the
prior (the YAML) and the first steps of IPOPT decide where along it the fit stops.

What a measurement can add: a blower-door test gives the infiltration rate. With the YAML regenerated at that
rate and the conductivity as the only declared parameter, the same fit recovers it.

In [ ]:
with timed("calibration with the infiltration measured"):
    calibrated_known, result_known = study.calibrate(known_infiltration=True)
print(
    f"{result_known.method}: {result_known.variables} decision variable, {result_known.iterations} iterations, "
    f"{result_known.solve_time:.0f} s, {result_known.status}"
)
known = pd.DataFrame(
    {
        "assumed (YAML)": {study.CONDUCTIVITY: spec.nominal[study.CONDUCTIVITY]},
        "calibrated": result_known.factors,
        "true": {study.CONDUCTIVITY: study.TRUTH[study.CONDUCTIVITY]},
    }
)
known["error %"] = 100 * (known["calibrated"] - known["true"]) / known["true"]
display(known.round(4))
display(result_known.metrics().round(3))

## Key takeaways

TAKEAWAYS

## Further reading

* `docs/calibration.md` for the groups and the problem; `mpctrano.yaml_parameters` for the annotations.
* Trano's `runtime` package addresses the same YAML fields by path for the Buildings models compiled with
  OpenModelica (`trano.runtime.specification`): the two conventions agree.